In [4]:
# W&B PE ablation grouped summary
import pandas as pd
import wandb
from typing import Any, Dict

api = wandb.Api()
project_path = "francisco-soto-u-universidad-de-chile/Paper-Periodic-LightCurves-Classification"

# Prefer filtering at API to reduce traffic
try:
    runs = api.runs(project_path, filters={"group": "DIFFT-PE-Ablation", "state": "finished"})
except Exception as e:
    print(f"W&B API error: {e}. Ensure WANDB_API_KEY is set and you have access.")
    runs = []

# Helper to read nested or dotted config keys
def get_cfg(cfg: Dict[str, Any], dotted: str, nested_path: list):
    # Try nested path first
    cur = cfg
    try:
        for k in nested_path:
            cur = cur[k]
        return cur
    except Exception:
        pass
    # Fallback to dotted keys
    if dotted in cfg:
        return cfg[dotted]
    return None

metric_candidates = [
    "test/f1",
    "val/f1",
    "test/f1_macro",
    "val/f1_macro",
    "best_f1",
]

rows = []
for run in runs:
    cfg = {k: v for k, v in run.config.items() if not str(k).startswith("_")}
    # Extract HPs from config
    kernel_size = get_cfg(cfg,
                          "model.lc_net.time_encoder.kernel_size",
                          ["model", "lc_net", "time_encoder", "kernel_size"]) 
    fusion_strategy = get_cfg(cfg,
                              "model.lc_net.time_encoder.fusion_strategy",
                              ["model", "lc_net", "time_encoder", "fusion_strategy"]) 
    reduced_size = get_cfg(cfg,
                           "model.lc_net.time_encoder.reduced_size_factor",
                           ["model", "lc_net", "time_encoder", "reduced_size_factor"]) 
    data_split = cfg.get("data.split")

    # Pick metric from summary
    summary = run.summary._json_dict
    metric_val = None
    for key in metric_candidates:
        if key in summary and isinstance(summary[key], (int, float)):
            metric_val = float(summary[key])
            break
    # Skip runs without required HPs or metric
    if kernel_size is None or fusion_strategy is None or reduced_size is None or metric_val is None:
        continue

    rows.append({
        "name": run.name,
        "kernel_size": kernel_size,
        "fusion_strategy": fusion_strategy,
        "reduced_size_factor": reduced_size,
        "data.split": data_split,
        "metric": metric_val,
    })

df = pd.DataFrame(rows)
if df.empty:
    print("No runs found or missing metrics/HPs for DIFFT-PE-Ablation.")
else:
    # Group by requested HPs
    grouped = (
        df.groupby(["kernel_size", "fusion_strategy", "reduced_size_factor"])  
          .agg(metric_mean=("metric", "mean"),
               metric_std=("metric", "std"),
               runs_count=("metric", "count"))
          .reset_index()
    )
    display(grouped.sort_values(["fusion_strategy", "kernel_size", "reduced_size_factor"]))


,kernel_size,fusion_strategy,reduced_size_factor,metric_mean,metric_std,runs_count
0,3,mlp,0.5,0.805405,0.004795,5
1,3,mlp,1.0,0.796905,0.001519,5
2,3,mlp,2.0,0.797993,0.004366,5
6,5,mlp,0.5,0.804465,0.002216,5
7,5,mlp,1.0,0.803293,0.004302,5
8,5,mlp,2.0,0.794468,0.006089,5
12,7,mlp,0.5,0.785665,0.001995,5
13,7,mlp,1.0,0.785742,0.004400,5
14,7,mlp,2.0,0.782232,0.003484,5
18,9,mlp,0.5,0.767109,0.004286,5


In [5]:
# W&B PE ablation: val/f1 at minimum val/loss
import pandas as pd
import wandb
from typing import Any, Dict

api = wandb.Api()
project_path = "francisco-soto-u-universidad-de-chile/Paper-Periodic-LightCurves-Classification"

try:
    runs = api.runs(project_path, filters={"group": "DIFFT-PE-Ablation", "state": "finished"})
except Exception as e:
    print(f"W&B API error: {e}. Ensure WANDB_API_KEY is set and you have access.")
    runs = []

# Helper to read nested or dotted config keys
def get_cfg(cfg: Dict[str, Any], dotted: str, nested_path: list):
    cur = cfg
    try:
        for k in nested_path:
            cur = cur[k]
        return cur
    except Exception:
        pass
    if dotted in cfg:
        return cfg[dotted]
    return None

rows = []
for run in runs:
    cfg = {k: v for k, v in run.config.items() if not str(k).startswith("_")}
    kernel_size = get_cfg(cfg, "model.lc_net.time_encoder.kernel_size", ["model", "lc_net", "time_encoder", "kernel_size"]) 
    fusion_strategy = get_cfg(cfg, "model.lc_net.time_encoder.fusion_strategy", ["model", "lc_net", "time_encoder", "fusion_strategy"]) 
    reduced_size = get_cfg(cfg, "model.lc_net.time_encoder.reduced_size_factor", ["model", "lc_net", "time_encoder", "reduced_size_factor"]) 
    data_split = cfg.get("data.split")

    # Fetch run history and select val/f1 at minimum val/loss
    hist_df = None
    try:
        hist_df = run.history(keys=["val/loss", "val/f1"])  # returns a pandas DataFrame
    except Exception as e:
        print(f"History fetch failed for {run.name}: {e}")
        continue

    if hist_df is None or hist_df.empty:
        continue

    dfh = hist_df[["val/loss", "val/f1"]].apply(pd.to_numeric, errors="coerce").dropna()
    if dfh.empty:
        continue

    idxmin = dfh["val/loss"].idxmin()
    val_f1_at_min_loss = float(dfh.loc[idxmin, "val/f1"]) if pd.notnull(dfh.loc[idxmin, "val/f1"]) else None
    if kernel_size is None or fusion_strategy is None or reduced_size is None or val_f1_at_min_loss is None:
        continue

    rows.append({
        "name": run.name,
        "kernel_size": kernel_size,
        "fusion_strategy": fusion_strategy,
        "reduced_size_factor": reduced_size,
        "data.split": data_split,
        "metric": val_f1_at_min_loss,
    })

df = pd.DataFrame(rows)
if df.empty:
    print("No runs with usable history (val/loss & val/f1) found for DIFFT-PE-Ablation.")
else:
    grouped = (
        df.groupby(["kernel_size", "fusion_strategy", "reduced_size_factor"])  
          .agg(metric_mean=("metric", "mean"),
               metric_std=("metric", "std"),
               runs_count=("metric", "count"))
          .reset_index()
    )
    display(grouped.sort_values(["fusion_strategy", "kernel_size", "reduced_size_factor"]))

,kernel_size,fusion_strategy,reduced_size_factor,metric_mean,metric_std,runs_count
0,3,mlp,0.5,0.803410,0.005812,5
1,3,mlp,1.0,0.797780,0.005907,5
2,3,mlp,2.0,0.794986,0.007227,5
6,5,mlp,0.5,0.796830,0.006645,5
7,5,mlp,1.0,0.796404,0.005726,5
8,5,mlp,2.0,0.795877,0.003731,5
12,7,mlp,0.5,0.784686,0.005071,5
13,7,mlp,1.0,0.785926,0.008846,5
14,7,mlp,2.0,0.783552,0.008106,5
18,9,mlp,0.5,0.765515,0.003222,5


In [9]:
# Permutation test (mlxtend) vs best group: val/f1 at min val/loss
import pandas as pd
import numpy as np
import wandb
from typing import Any, Dict
try:
    from mlxtend.evaluate import permutation_test
except Exception:
    print("mlxtend not installed. Please install with: pip install mlxtend")
    permutation_test = None

api = wandb.Api()
project_path = "francisco-soto-u-universidad-de-chile/Paper-Periodic-LightCurves-Classification"

try:
    runs = api.runs(project_path, filters={"group": "DIFFT-PE-Ablation", "state": "finished"})
except Exception as e:
    print(f"W&B API error: {e}. Ensure WANDB_API_KEY is set and you have access.")
    runs = []

def get_cfg(cfg: Dict[str, Any], dotted: str, nested_path: list):
    cur = cfg
    try:
        for k in nested_path:
            cur = cur[k]
        return cur
    except Exception:
        pass
    if dotted in cfg:
        return cfg[dotted]
    return None

rows = []
for run in runs:
    cfg = {k: v for k, v in run.config.items() if not str(k).startswith("_")}
    kernel_size = get_cfg(cfg, "model.lc_net.time_encoder.kernel_size", ["model", "lc_net", "time_encoder", "kernel_size"])
    fusion_strategy = get_cfg(cfg, "model.lc_net.time_encoder.fusion_strategy", ["model", "lc_net", "time_encoder", "fusion_strategy"])
    reduced_size = get_cfg(cfg, "model.lc_net.time_encoder.reduced_size_factor", ["model", "lc_net", "time_encoder", "reduced_size_factor"])
    try:
        hist_df = run.history(keys=["val/loss", "val/f1"])
    except Exception as e:
        print(f"History fetch failed for {run.name}: {e}")
        continue
    if hist_df is None or hist_df.empty:
        continue
    dfh = hist_df[["val/loss", "val/f1"]].apply(pd.to_numeric, errors="coerce").dropna()
    if dfh.empty:
        continue
    idxmin = dfh["val/loss"].idxmin()
    val_f1_at_min_loss = float(dfh.loc[idxmin, "val/f1"]) if pd.notnull(dfh.loc[idxmin, "val/f1"]) else None
    if kernel_size is None or fusion_strategy is None or reduced_size is None or val_f1_at_min_loss is None:
        continue
    rows.append({
        "name": run.name,
        "kernel_size": kernel_size,
        "fusion_strategy": fusion_strategy,
        "reduced_size_factor": reduced_size,
        "metric": val_f1_at_min_loss,
    })

df = pd.DataFrame(rows)
if df.empty:
    print("No runs with usable history (val/loss & val/f1) found for DIFFT-PE-Ablation.")
elif permutation_test is None:
    print("Permutation test unavailable; install mlxtend and re-run.")
else:
    group_cols = ["kernel_size", "fusion_strategy", "reduced_size_factor"]
    means = df.groupby(group_cols)["metric"].mean()
    if means.empty:
        print("No grouped means available; check input data.")
    else:
        best_key = means.idxmax()
        best_mean = float(means.loc[best_key])
        best_arr = df[(df[group_cols[0]]==best_key[0]) & (df[group_cols[1]]==best_key[1]) & (df[group_cols[2]]==best_key[2])]["metric"].to_numpy()
        results = []
        for key, mean_val in means.items():
            if key == best_key:
                continue
            arr = df[(df[group_cols[0]]==key[0]) & (df[group_cols[1]]==key[1]) & (df[group_cols[2]]==key[2])]["metric"].to_numpy()
            if len(arr) == 0 or len(best_arr) == 0:
                continue
            try:
                pval = permutation_test(
                best_arr, 
                arr,
                func='x_mean > y_mean',
                method='approximate',
                num_rounds=10000,
                paired=True
            )
            except Exception as e:
                print(f"Permutation test failed for {key}: {e}")
                continue
            results.append({
                "best_kernel_size": best_key[0],
                "best_fusion_strategy": best_key[1],
                "best_reduced_size_factor": best_key[2],
                "best_mean": best_mean,
                "kernel_size": key[0],
                "fusion_strategy": key[1],
                "reduced_size_factor": key[2],
                "other_mean": float(mean_val),
                "mean_diff": best_mean - float(mean_val),
                "p_value": float(pval)
            })
        res_df = pd.DataFrame(results)
        if res_df.empty:
            print("No permutation test results to show.")
        else:
            res_df_sorted = res_df.sort_values(["fusion_strategy", "kernel_size", "reduced_size_factor"])
            display(res_df_sorted)
            try:
                out_path = "paper_notebooks/csv/pe_ablation_permutation_vs_best.csv"
                res_df_sorted.to_csv(out_path, index=False)
                print(f"Saved permutation results to {out_path}")
            except Exception as e:
                print(f"Failed to save CSV: {e}")

,best_kernel_size,best_fusion_strategy,best_reduced_size_factor,best_mean,kernel_size,fusion_strategy,reduced_size_factor,other_mean,mean_diff,p_value
0,3,mlp,0.5,0.80341,3,mlp,1.0,0.797780,0.005630,0.093491
1,3,mlp,0.5,0.80341,3,mlp,2.0,0.794986,0.008424,0.030397
5,3,mlp,0.5,0.80341,5,mlp,0.5,0.796830,0.006580,0.063294
6,3,mlp,0.5,0.80341,5,mlp,1.0,0.796404,0.007006,0.093591
7,3,mlp,0.5,0.80341,5,mlp,2.0,0.795877,0.007532,0.096590
11,3,mlp,0.5,0.80341,7,mlp,0.5,0.784686,0.018723,0.030297
12,3,mlp,0.5,0.80341,7,mlp,1.0,0.785926,0.017483,0.029597
13,3,mlp,0.5,0.80341,7,mlp,2.0,0.783552,0.019857,0.031097
17,3,mlp,0.5,0.80341,9,mlp,0.5,0.765515,0.037895,0.033197
18,3,mlp,0.5,0.80341,9,mlp,1.0,0.768000,0.035409,0.032497


Failed to save CSV: Cannot save file into a non-existent directory: 'paper_notebooks/csv'


In [10]:

# Generate LaTeX table for ablation results
import pandas as pd
import numpy as np
import wandb
from typing import Any, Dict

api = wandb.Api()
project_path = "francisco-soto-u-universidad-de-chile/Paper-Periodic-LightCurves-Classification"

try:
    runs = api.runs(project_path, filters={"group": "DIFFT-PE-Ablation", "state": "finished"})
except Exception as e:
    print(f"W&B API error: {e}")
    runs = []

def get_cfg(cfg: Dict[str, Any], dotted: str, nested_path: list):
    cur = cfg
    try:
        for k in nested_path:
            cur = cur[k]
        return cur
    except Exception:
        pass
    if dotted in cfg:
        return cfg[dotted]
    return None

rows = []
for run in runs:
    cfg = {k: v for k, v in run.config.items() if not str(k).startswith("_")}
    kernel_size = get_cfg(cfg, "model.lc_net.time_encoder.kernel_size", ["model", "lc_net", "time_encoder", "kernel_size"])
    fusion_strategy = get_cfg(cfg, "model.lc_net.time_encoder.fusion_strategy", ["model", "lc_net", "time_encoder", "fusion_strategy"])
    reduced_size = get_cfg(cfg, "model.lc_net.time_encoder.reduced_size_factor", ["model", "lc_net", "time_encoder", "reduced_size_factor"])
    try:
        hist_df = run.history(keys=["val/loss", "val/f1"])
    except Exception:
        continue
    if hist_df is None or hist_df.empty:
        continue
    dfh = hist_df[["val/loss", "val/f1"]].apply(pd.to_numeric, errors="coerce").dropna()
    if dfh.empty:
        continue
    idxmin = dfh["val/loss"].idxmin()
    val_f1_at_min_loss = float(dfh.loc[idxmin, "val/f1"]) if pd.notnull(dfh.loc[idxmin, "val/f1"]) else None
    if kernel_size is None or fusion_strategy is None or reduced_size is None or val_f1_at_min_loss is None:
        continue
    rows.append({
        "kernel_size": kernel_size,
        "fusion_strategy": fusion_strategy,
        "reduced_size_factor": reduced_size,
        "metric": val_f1_at_min_loss,
    })

df = pd.DataFrame(rows)
if not df.empty:
    # Compute stats by group
    grouped_stats = df.groupby(["kernel_size", "fusion_strategy", "reduced_size_factor"])["metric"].agg(["mean", "std", "count"]).reset_index()
    
    # Load permutation results to identify significance
    try:
        perm_df = pd.read_csv("paper_notebooks/csv/pe_ablation_permutation_vs_best.csv")
        sig_pairs = set()
        for _, row in perm_df.iterrows():
            if row["p_value"] > 0.05:  # Not significantly different from best
                sig_pairs.add((
                    int(row["kernel_size"]),
                    row["fusion_strategy"],
                    float(row["reduced_size_factor"])
                ))
    except Exception:
        sig_pairs = set()
    
    # Create pivot for LaTeX table
    latex_lines = []
    latex_lines.append(r"\begin{table}[h]")
    latex_lines.append(r"\centering")
    latex_lines.append(r"\caption{PE Ablation Study: Classification performance (mean $\pm$ std). Bold = not significantly different from best.}")
    latex_lines.append(r"\label{tab:pe_ablation}")
    latex_lines.append(r"\begin{tabular}{|c|c|c|c|}")
    latex_lines.append(r"\hline")
    latex_lines.append(r"{Kernel} & {Embedding} & \multicolumn{2}{c|}{Fusion Strategy} \\")
    latex_lines.append(r"\cline{3-4}")
    latex_lines.append(r" Size & Rate & MLP & Simple \\")
    latex_lines.append(r"\hline")
    
    # Sort and iterate
    kernel_sizes = sorted(grouped_stats["kernel_size"].unique())
    reduced_sizes = sorted(grouped_stats["reduced_size_factor"].unique())
    
    for ks in kernel_sizes:
        for rs in reduced_sizes:
            mlp_row = grouped_stats[
                (grouped_stats["kernel_size"] == ks) & 
                (grouped_stats["reduced_size_factor"] == rs) &
                (grouped_stats["fusion_strategy"] == "mlp")
            ]
            simple_row = grouped_stats[
                (grouped_stats["kernel_size"] == ks) & 
                (grouped_stats["reduced_size_factor"] == rs) &
                (grouped_stats["fusion_strategy"] == "simple")
            ]
            
            mlp_str = ""
            if not mlp_row.empty:
                m, s = mlp_row.iloc[0]["mean"] * 100, mlp_row.iloc[0]["std"] * 100
                is_sig = (int(ks), "mlp", float(rs)) in sig_pairs
                mlp_str = rf"{m:.2f} $\pm$ {s:.2f}"
                if is_sig:
                    mlp_str = rf"\textbf{{{mlp_str}}}"
            
            simple_str = ""
            if not simple_row.empty:
                m, s = simple_row.iloc[0]["mean"] * 100, simple_row.iloc[0]["std"] * 100
                is_sig = (int(ks), "simple", float(rs)) in sig_pairs
                simple_str = rf"{m:.2f} $\pm$ {s:.2f}"
                if is_sig:
                    simple_str = rf"\textbf{{{simple_str}}}"
            
            latex_lines.append(f"{int(ks)} & {rs} & {mlp_str} & {simple_str} \\\\")
    
    latex_lines.append(r"\hline")
    latex_lines.append(r"\end{tabular}")
    latex_lines.append(r"\end{table}")
    
    latex_table = "\n".join(latex_lines)
    print(latex_table)
    
    # Save to file
    try:
        with open("paper_notebooks/pe_ablation_table.tex", "w") as f:
            f.write(latex_table)
        print("\nLaTeX table saved to paper_notebooks/pe_ablation_table.tex")
    except Exception as e:
        print(f"Failed to save LaTeX table: {e}")


\begin{table}[h]
\centering
\caption{PE Ablation Study: Classification performance (mean $\pm$ std). Bold = not significantly different from best.}
\label{tab:pe_ablation}
\begin{tabular}{|c|c|c|c|}
\hline
{Kernel} & {Embedding} & \multicolumn{2}{c|}{Fusion Strategy} \\
\cline{3-4}
 Size & Rate & MLP & Simple \\
\hline
3 & 0.5 & 80.34 $\pm$ 0.58 & 75.66 $\pm$ 0.40 \\
3 & 1.0 & 79.78 $\pm$ 0.59 & 76.38 $\pm$ 0.64 \\
3 & 2.0 & 79.50 $\pm$ 0.72 & 75.65 $\pm$ 0.51 \\
5 & 0.5 & 79.68 $\pm$ 0.66 & 74.87 $\pm$ 0.40 \\
5 & 1.0 & 79.64 $\pm$ 0.57 & 75.13 $\pm$ 1.13 \\
5 & 2.0 & 79.59 $\pm$ 0.37 & 74.45 $\pm$ 0.74 \\
7 & 0.5 & 78.47 $\pm$ 0.51 & 74.22 $\pm$ 0.51 \\
7 & 1.0 & 78.59 $\pm$ 0.88 & 74.04 $\pm$ 0.81 \\
7 & 2.0 & 78.36 $\pm$ 0.81 & 74.18 $\pm$ 0.50 \\
9 & 0.5 & 76.55 $\pm$ 0.32 & 73.69 $\pm$ 0.60 \\
9 & 1.0 & 76.80 $\pm$ 0.55 & 73.33 $\pm$ 1.02 \\
9 & 2.0 & 76.79 $\pm$ 0.69 & 73.46 $\pm$ 0.68 \\
\hline
\end{tabular}
\end{table}
Failed to save LaTeX table: [Errno 2] No such file or dire

In [ ]:

# Compare periodic feature importance with dict_info features
import pandas as pd
import yaml

# Load dict_info to get available features
dict_info_path = "data/alerce_data/final/dict_info.yaml"
with open(dict_info_path, 'r') as f:
    dict_info = yaml.safe_load(f)

dict_features = set(dict_info.get('feat_cols', []))
dict_metadata = set(dict_info.get('md_cols', []))
print(f"Features in dict_info: {len(dict_features)}")
print(f"Metadata in dict_info: {len(dict_metadata)}\n")

# Load periodic features importance (from previous cell output)
periodic_features_text = """W1-W2? 0.094 W1-W2? 0.109 Multiband period 0.089 sgscore1‡ 0.053 sgscore1‡ 0.058 
g-W2? 0.062 g-W2? 0.046 g-W2? 0.031 g-W3? 0.028 g-W3? 0.037 g-W3? 0.031
r-W2? 0.049 r-W2? 0.034 r-W3? 0.023 r-W3? 0.016
(g-r) mean corr 0.048 (g-r) max corr 0.030 (g-r) mean 0.027 (g-r) max 0.025 (g-r) max 0.015 (g-r) mean 0.012 (g-r) max corr 0.035 (g-r) mean corr 0.022 (g-r) mean 0.025
positive fraction 2 0.050 positive fraction 1 0.048
SPM t0 1 0.033 SPM t0 2 0.022 SPM gamma 2 0.029 SPM gamma 1 0.017 SPM gamma 1 0.024
SPM tau rise 2 0.028 SPM tau rise 1 0.025 SPM tau rise 1 0.035 SPM tau rise 2 0.023
LinearTrend 2 0.032 LinearTrend 2 0.019 LinearTrend 1 0.013 LinearTrend 1 0.012
SPM chi 1 0.031 SPM chi 2 0.020
ExcessVar 2 0.033 ExcessVar 1 0.017 ExcessVar 1 0.011
Meanvariance 2 0.026 Meanvariance 1 0.016 Meanvariance 2 0.012
Amplitude 1 0.017 Amplitude 2 0.022 Amplitude 2 0.007
IAR phi 1 0.022 IAR phi 1 0.010 IAR phi 1 0.009 IAR phi 2 0.007 IAR phi 2 0.008
GP DRW tau 1 0.023 GP DRW sigma 1 0.015 GP DRW tau 2 0.012 GP DRW tau 1 0.007
Std 2 0.015 Std 1 0.015
PercentAmplitude 2 0.013 PercentAmplitude 1 0.012
W2-W3? 0.025 W2-W3? 0.022 W2-W3? 0.009
Rcs 2 0.013
GP DRW sigma 2 0.013 GP DRW sigma 2 0.009
SPM A 2 0.023 SPM A 1 0.019 SPM A 2 0.014 SPM A 1 0.012 SPM A 1 0.009
SPM tau fall 1 0.013 SPM tau fall 2 0.017 SPM tau fall 1 0.011
MHPS low 1 0.013 MHPS ratio 1 0.011 MHPS ratio 2 0.010 MHPS low 2 0.008 MHPS low 1 0.006 MHPS high 1 0.007
Power rate 2 0.009
Pvar 2 0.010 Pvar 2 0.007
Skew 2 0.009 Gskew 1 0.009
SF ML amplitude 1 0.009 SF ML amplitude 2 0.009 SF ML gamma 1 0.008
AndersonDarling 2 0.018
delta mag fid 2 0.024 delta mag fid 1 0.016 delta mag fid 1 0.009
SPM beta 2 0.011 SPM beta 1 0.009"""

# Normalize feature names for matching
def normalize_name(name):
    """Normalize feature name to match dict format"""
    if not name:
        return ""
    # Replace special characters and normalize spacing
    name = name.replace('?', '').replace('‡', '').strip()
    # Handle color names: g-W2 -> g_W2
    name = name.replace('-', '_')
    # Handle parentheses: (g-r) -> g_r
    name = name.replace('(', '').replace(')', '')
    # SPM names
    name = name.replace('SPM_t0', 'SPM_t0')
    name = name.replace('SPM_tau_rise', 'SPM_tau_rise')
    name = name.replace('SPM_tau_fall', 'SPM_tau_fall')
    name = name.replace('SPM_gamma', 'SPM_gamma')
    name = name.replace('SPM_chi', 'SPM_chi')
    name = name.replace('SPM_A', 'SPM_A')
    name = name.replace('SPM_beta', 'SPM_beta')
    # MultiWord features
    name = name.replace('Multiband_period', 'Multiband_period_12')
    name = name.replace('LinearTrend', 'LinearTrend')
    name = name.replace('ExcessVar', 'ExcessVar')
    name = name.replace('Meanvariance', 'Meanvariance')
    name = name.replace('PercentAmplitude', 'PercentAmplitude')
    name = name.replace('IAR_phi', 'IAR_phi')
    name = name.replace('GP_DRW_tau', 'GP_DRW_tau')
    name = name.replace('GP_DRW_sigma', 'GP_DRW_sigma')
    name = name.replace('MHPS_low', 'MHPS_low')
    name = name.replace('MHPS_high', 'MHPS_high')
    name = name.replace('MHPS_ratio', 'MHPS_ratio')
    name = name.replace('SF_ML', 'SF_ML')
    name = name.replace('AndersonDarling', 'AndersonDarling')
    name = name.replace('Amplitude', 'Amplitude')
    name = name.replace('Pvar', 'Pvar')
    name = name.replace('Rcs', 'Rcs')
    name = name.replace('Skew', 'Skew')
    name = name.replace('Gskew', 'Gskew')
    name = name.replace('Std', 'Std')
    name = name.replace('Power_rate', 'Power_rate')
    name = name.replace('positive_fraction', 'positive_fraction')
    # Metadata
    name = name.replace('sgscore1', 'sgscore1')
    return name.upper()

# Parse periodic features
periodic_list = []
tokens = periodic_features_text.split()
i = 0
while i < len(tokens):
    if i + 1 < len(tokens):
        try:
            score = float(tokens[i + 1])
            feature_name = tokens[i]
            norm_name = normalize_name(feature_name)
            periodic_list.append({
                "original_name": feature_name,
                "normalized_name": norm_name,
                "importance": score
            })
            i += 2
        except ValueError:
            i += 1
    else:
        i += 1

df_periodic = pd.DataFrame(periodic_list)
df_periodic = df_periodic.drop_duplicates(subset=['normalized_name'], keep='first').sort_values('importance', ascending=False)

# Match with dict_info features
matches = []
for idx, row in df_periodic.iterrows():
    norm_name = row['normalized_name']
    # Try different suffix patterns (band 1, 2, or multi-band)
    for suffix in ['_1', '_2', '_12', '']:
        full_name = f"{norm_name.lower()}{suffix}"
        if full_name in dict_features:
            matches.append({
                "original_name": row['original_name'],
                "dict_feature": full_name,
                "importance": row['importance'],
                "matched": True
            })
            break
    else:
        # Check metadata
        if row['normalized_name'].lower() in dict_metadata:
            matches.append({
                "original_name": row['original_name'],
                "dict_feature": row['normalized_name'].lower(),
                "importance": row['importance'],
                "matched": True
            })
        else:
            matches.append({
                "original_name": row['original_name'],
                "dict_feature": "NOT FOUND",
                "importance": row['importance'],
                "matched": False
            })

df_matches = pd.DataFrame(matches)

# Display top matched features
print("Top 30 Periodic Features (Ranked by Importance):")
print("=" * 80)
top_matched = df_matches[df_matches['matched']].head(30)
display(top_matched[['original_name', 'dict_feature', 'importance']])

print(f"\n\nSummary:")
print(f"Total unique periodic features: {len(df_periodic)}")
print(f"Matched with dict_info: {df_matches['matched'].sum()}")
print(f"Not found in dict_info: {(~df_matches['matched']).sum()}")

# Save matched features
try:
    df_matches[df_matches['matched']].to_csv("paper_notebooks/csv/periodic_features_matched_dict_info.csv", index=False)
    print(f"\nSaved matched features to paper_notebooks/csv/periodic_features_matched_dict_info.csv")
except Exception as e:
    print(f"Failed to save: {e}")


In [1]:
# Compare groups containing 'vicreg' and 'linear': mean test F1 and p-value
import pandas as pd
import numpy as np
import wandb
from typing import Any, Dict

# Optional permutation test; fallback to Welch's t-test if unavailable
try:
    from mlxtend.evaluate import permutation_test
except Exception:
    permutation_test = None
from scipy.stats import ttest_ind

api = wandb.Api()
project_path = "francisco-soto-u-universidad-de-chile/Paper-Periodic-LightCurves-Classification"

# Fetch finished runs (keep API-side filtering minimal, filter groups in Python)
try:
    runs = api.runs(project_path, filters={"state": "finished"})
except Exception as e:
    print(f"W&B API error: {e}. Ensure WANDB_API_KEY is set and you have access.")
    runs = []

rows = []
for run in runs:
    grp = getattr(run, "group", None)
    if not isinstance(grp, str):
        continue
    grp_low = grp.lower()
    if ("vicreg" not in grp_low) and ("linear" not in grp_low):
        continue
    summary = run.summary._json_dict
    test_f1 = summary.get("test/f1")
    if isinstance(test_f1, (int, float)):
        rows.append({
            "run_name": run.name,
            "group": grp,
            "supergroup": "vicreg" if "vicreg" in grp_low else ("linear" if "linear" in grp_low else "other"),
            "test_f1": float(test_f1),
        })

if not rows:
    print("No finished runs found with groups containing 'vicreg' or 'linear' and a numeric test/f1 in summary.")
else:
    df = pd.DataFrame(rows)
    # Keep only the two supergroups of interest
    df = df[df["supergroup"].isin(["vicreg", "linear"])]

    # Aggregate stats
    grouped = (df.groupby(["supergroup"])  
                 .agg(mean_test_f1=("test_f1", "mean"),
                      std_test_f1=("test_f1", "std"),
                      n=("test_f1", "count"))
                 .reset_index())
    display(grouped)

    # Prepare arrays for significance test
    arr_v = df[df["supergroup"] == "vicreg"]["test_f1"].to_numpy()
    arr_l = df[df["supergroup"] == "linear"]["test_f1"].to_numpy()

    p_perm = None
    if permutation_test is not None and len(arr_v) > 0 and len(arr_l) > 0:
        try:
            p_perm = float(permutation_test(arr_v, arr_l, func='x_mean > y_mean', method='approximate', num_rounds=10000, paired=False))
        except Exception:
            p_perm = None

    # Fallback to Welch's t-test (two-sided); also report one-sided if desired
    p_ttest = None
    one_sided = None
    if len(arr_v) > 1 and len(arr_l) > 1:
        try:
            tstat, p_two = ttest_ind(arr_v, arr_l, equal_var=False, nan_policy='omit')
            p_ttest = float(p_two)
            # Optional one-sided p-value (vicreg > linear)
            one_sided = float(p_two / 2) if (np.mean(arr_v) > np.mean(arr_l)) else float(1 - p_two / 2)
        except Exception:
            pass

    print("\nSignificance between vicreg and linear (test/f1):")
    if p_perm is not None:
        print(f"Permutation test (vicreg > linear) p-value: {p_perm:.6f}")
    else:
        print("Permutation test unavailable or failed.")
    if p_ttest is not None:
        print(f"Welch t-test (two-sided) p-value: {p_ttest:.6f}")
        if one_sided is not None:
            print(f"Welch t-test (one-sided, vicreg > linear) p-value: {one_sided:.6f}")

    # Save detailed rows and summary
    try:
        out_dir = Path("paper_notebooks/csv")
        out_dir.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_dir / "vicreg_linear_runs_test_f1.csv", index=False)
        grouped.to_csv(out_dir / "vicreg_linear_summary_test_f1.csv", index=False)
        print("\nSaved:\n - paper_notebooks/csv/vicreg_linear_runs_test_f1.csv\n - paper_notebooks/csv/vicreg_linear_summary_test_f1.csv")
    except Exception as e:
        print(f"Failed to save CSVs: {e}")


,supergroup,mean_test_f1,std_test_f1,n
0,vicreg,0.554585,0.095938,100



Significance between vicreg and linear (test/f1):
Permutation test unavailable or failed.
Failed to save CSVs: name 'Path' is not defined
